# Категории: основные показатели

Пример расчета базовых показателей по категориям

## Описание задачи и условий расчета
- Период: весь доступный период
- Категории: все доступные категории
- Статистики: количество запросов (тыс), распределение запросов по типу ресурса (%)

### Инициализация

Импортируем библиотеки, которые понадобятся для работы.\
Выполните следующую ячейку, для этого перейдите в нее и нажмите Shift + Enter

In [ ]:
import pandas as pd
import numpy as np

from fileapiback import *

### Формирование задания

Зададим условия для расчета\
*см. подробнее формат условий в блокноте [help.ipynb](help.ipynb)*

In [ ]:
# укажите период в формате:
# None (все месяцы с активностью в архиве) или ['YYYY-MM-01', 'YYYY-MM-01'] (конкретные месяцы)
period = None

# укажите категории для фильтрации в формате:
# None (все категории) или ['Смартфоны', 'Дезодоранты']
category_filter = None

# укажите, нужна ли разбивка ecom для показателя "Распределение запросов по типу ресурса": 
# да - True, нет - False
flag_filter = False

### Выполнение задания
- Визуализация динамики количества запросов
- Расчет
  - количества запросов в разбивке по категориям по месяцам (тыс)
  - распределения запросов в разбивке по категориям по типу ресурса по месяцам (%)

In [ ]:
# формирование pandas.DataFrame

obj = FileApi()

# создаем pandas.DataFrame
data = obj.create_pdDF(note='category', period=period, category_filter=category_filter, flag_filter=flag_filter)

In [ ]:
# расчет количества запросов в разбивке по категориям по месяцам

qcnt = (data
        .groupby(['month', 'Category'])['Weight'] # группируем по месяцу, категории и считаем количество запросов
        .sum()  # сумма по группе
        .reset_index(name='QCnt000')  # переименование столбца
        .sort_values(by=['month', 'QCnt000'], ascending=[True, False])  # сортировка
    )

# округляем показатели
qcnt['QCnt000'] = (qcnt['QCnt000'] / 1000)

In [ ]:
# построение графика
obj.draw_diagram(df=qcnt, period=period, col='QCnt000', brand_cat='Category')

In [ ]:
# распределение запросов в разбивке по категориям по типу ресурса

# считаем количество запросов
res_share = (data
             .groupby(['month', 'Category', 'ResourceType'])['Weight'] # группируем по месяцу, категории, типу ресерса
             .sum()
             .reset_index(name='QCnt')
            )

res_share['QCnt_all'] = res_share.groupby(['month', 'Category'])['QCnt'].transform('sum') # добавляем общую сумму
res_share['%'] = (res_share['QCnt'] / res_share['QCnt_all'] * 100).round(1) # вычисляем процент

# выбираем нужные столбцы и сортируем
res_share = (res_share[['month', 'Category', 'ResourceType', '%']]
            .sort_values(['month', 'Category', '%'], ascending=[True, True, False]))

## Экспорт в Excel
По умолчанию файл 'categories-metrics-YYYYMMDD_HHMMSS.xlsx' сохраняется в ту же директорию, где лежат все блокноты

In [ ]:
time_now = datetime.now().strftime('%Y%m%d_%H%M%S')

with pd.ExcelWriter(f'categories-metrics-{time_now}.xlsx') as writer:
    qcnt.to_excel(writer, 'qcnt', index=False)
    res_share.to_excel(writer, 'res_share', index=False)